In [1]:
from pathlib import Path
from getpass import getpass
from typing import Literal
import json
import os
import time

import pandas as pd
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
INPUT_PATH = PROJECT_ROOT / 'results/data_quality/hplt_en_uz/manual_review_200.csv'
OUTPUT_PATH = PROJECT_ROOT / 'results/data_quality/hplt_en_uz/manual_review_200_llm.csv'
MODEL = os.getenv('AUDIT_MODEL', 'qwen-max').strip()
BASE_URL = os.getenv('LLM_BASE_URL', '').strip() or 'https://dashscope.aliyuncs.com/compatible-mode/v1'
API_KEY = os.getenv('LLM_API_KEY', '').strip() or os.getenv('OPENAI_API_KEY', '').strip()
BATCH_SIZE = 5
MAX_ROWS = 200
MAX_RETRIES = 4
REQUEST_DELAY_SECONDS = 0.2

if not MODEL:
    MODEL = input('Model name: ').strip()
if not API_KEY:
    API_KEY = getpass('API key: ').strip()
assert MODEL, 'Model name is required.'
assert API_KEY, 'API key is required.'
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, timeout=90.0, max_retries=0)
print('Model:', MODEL)
print('Base URL:', BASE_URL or 'provider default')
print('Input:', INPUT_PATH)
print('Output:', OUTPUT_PATH)

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: qwen-max
Base URL: https://dashscope.aliyuncs.com/compatible-mode/v1
Input: D:\dev\projects\fourlang_translation\results\data_quality\hplt_en_uz\manual_review_200.csv
Output: D:\dev\projects\fourlang_translation\results\data_quality\hplt_en_uz\manual_review_200_llm.csv


In [2]:
source_df = pd.read_csv(INPUT_PATH, keep_default_na=False).head(MAX_ROWS).copy()
required = {'pair_id', 'en', 'uz'}
missing = required - set(source_df.columns)
assert not missing, f'Missing columns: {sorted(missing)}'
result_columns = [
    'llm_label', 'llm_confidence', 'llm_adequacy',
    'llm_en_language_ok', 'llm_uz_language_ok', 'llm_reason',
    'llm_model', 'llm_error',
]
if OUTPUT_PATH.exists():
    old = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
    old_results = old[['pair_id'] + [c for c in result_columns if c in old.columns]].drop_duplicates('pair_id', keep='last')
    work_df = source_df.drop(columns=[c for c in result_columns if c in source_df.columns], errors='ignore').merge(old_results, on='pair_id', how='left')
else:
    work_df = source_df.copy()
for col in result_columns:
    if col not in work_df.columns:
        work_df[col] = ''
    # 结果列统一转为 object:需同时容纳 str / float / bool,
    # 避免 StringDtype 列拒绝写入数值(confidence/adequacy)导致整批失败
    work_df[col] = work_df[col].fillna('').astype(object)
# 上次审计失败的残留行(只写入了部分字段)重置为待审,
# 避免 completed_mask 把残留了 llm_label 的行误判为已完成
error_mask = work_df['llm_error'].fillna('').astype(str).str.strip() != ''
if error_mask.any():
    work_df.loc[error_mask, result_columns] = ''
    print(f'Reset {int(error_mask.sum())} rows with previous audit errors.')
completed_mask = work_df['llm_label'].isin(['correct', 'minor', 'wrong', 'junk'])
print(f'Rows: {len(work_df)}, completed: {int(completed_mask.sum())}, pending: {int((~completed_mask).sum())}')
display(work_df[['pair_id', 'sample_group', 'en', 'uz']].head())


Reset 200 rows with previous audit errors.
Rows: 200, completed: 0, pending: 200


,pair_id,sample_group,en,uz
0,hplt_00002652,high_risk,It's also pretty incredible that we've managed...,"Lekin, sizlardan 99%, ehtimol, biz uchun mosli..."
1,hplt_00003570,high_risk,"Retrieved March 9, 2021.",Qaraldi: 8-mart 2021-yil.
2,hplt_00002549,high_risk,Preset support.,Oldindan o'rnatilgan qo'llab -quvvatlash .
3,hplt_00005357,random_control,Unlike chat rouletteson other platforms you ca...,"Ruletka chatidan farqli o'laroq, siz tasodifiy..."
4,hplt_00007972,random_control,The journey will take about 5 minutes.,Safar taxminan 5 daqiqa davom etadi.


In [3]:
class AuditItem(BaseModel):
    pair_id: str
    label: Literal['correct', 'minor', 'wrong', 'junk']
    confidence: float = Field(ge=0.0, le=1.0)
    adequacy: int = Field(ge=0, le=4)
    en_language_ok: bool
    uz_language_ok: bool
    reason: str = Field(min_length=1, max_length=300)

class AuditBatch(BaseModel):
    items: list[AuditItem]

SYSTEM_PROMPT = '''You are a strict bilingual English-Uzbek parallel-corpus auditor.
Judge whether each Uzbek sentence faithfully translates its English sentence.
Do not reward topical similarity when facts, entities, numbers, negation, or meaning differ.
Labels:
- correct: meaning is faithfully preserved; naturalness issues are negligible.
- minor: core meaning is preserved but wording is awkward or a non-critical detail is omitted/added.
- wrong: mistranslation, important omission/addition, contradiction, or sentences are not translations.
- junk: corrupted text, unusable fragments, wrong language, or severe web noise.
Adequacy: 4=fully faithful, 3=mostly faithful, 2=partly faithful, 1=mostly unrelated, 0=unrelated/junk.
Use confidence conservatively. Give the reason in concise Chinese.
Return one result per pair_id. Return JSON only with this shape:
{"items":[{"pair_id":"...","label":"correct|minor|wrong|junk","confidence":0.0,"adequacy":0,"en_language_ok":true,"uz_language_ok":true,"reason":"..."}]}'''

def parse_json_object(text):
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text, flags=re.IGNORECASE)
    return json.loads(text)

import re

def audit_batch(batch_df):
    payload = {
        'pairs': [
            {'pair_id': str(row.pair_id), 'english': str(row.en), 'uzbek': str(row.uz)}
            for row in batch_df.itertuples(index=False)
        ]
    }
    expected_ids = set(batch_df['pair_id'].astype(str))
    last_error = None
    for attempt in range(MAX_RETRIES):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': json.dumps(payload, ensure_ascii=False)},
                ],
                response_format={'type': 'json_object'},
            )
            content = response.choices[0].message.content or ''
            parsed = AuditBatch.model_validate(parse_json_object(content))
            received_ids = {item.pair_id for item in parsed.items}
            if received_ids != expected_ids:
                raise ValueError(f'pair_id mismatch: expected {expected_ids}, received {received_ids}')
            return parsed.items
        except Exception as exc:
            last_error = exc
            if getattr(exc, 'status_code', None) in {400, 401, 403, 404}:
                raise RuntimeError(f'Non-retryable API error: {exc}') from exc
            if attempt + 1 < MAX_RETRIES:
                time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f'Audit failed after {MAX_RETRIES} attempts: {last_error}')

In [4]:
test_batch = work_df[~completed_mask].head(min(BATCH_SIZE, int((~completed_mask).sum())))
if len(test_batch):
    test_result = audit_batch(test_batch)
    display(pd.DataFrame([item.model_dump() for item in test_result]))
else:
    print('No pending rows.')

,pair_id,label,confidence,adequacy,en_language_ok,uz_language_ok,reason
0,hplt_00002652,wrong,0.9,1,True,False,乌兹别克语句子的意思与英语句子完全不同，包含了许多英语句子中没有的信息。
1,hplt_00003570,minor,0.8,3,True,True,日期的翻译有些微小差异（3月9日变成了8日），但核心意思基本一致。
2,hplt_00002549,correct,0.9,4,True,True,预设支持的意思被准确翻译了。
3,hplt_00005357,correct,0.9,4,True,True,句子的意思被准确翻译了。
4,hplt_00007972,correct,0.9,4,True,True,句子的意思被准确翻译了。


In [5]:
RUN_AUDIT = True

def save_progress(frame):
    temp_path = OUTPUT_PATH.with_suffix('.tmp.csv')
    frame.to_csv(temp_path, index=False, encoding='utf-8-sig')
    temp_path.replace(OUTPUT_PATH)

if not RUN_AUDIT:
    print('Batch audit is disabled. Review the test batch above, then set RUN_AUDIT = True.')
else:
    completed_mask = work_df['llm_label'].isin(['correct', 'minor', 'wrong', 'junk'])
    pending_indices = work_df.index[~completed_mask].tolist()
    batches = [pending_indices[i:i + BATCH_SIZE] for i in range(0, len(pending_indices), BATCH_SIZE)]
    for indices in tqdm(batches, desc='LLM audit batches'):
        batch_df = work_df.loc[indices]
        try:
            items = audit_batch(batch_df)
            by_id = {item.pair_id: item for item in items}
            for idx in indices:
                item = by_id[str(work_df.at[idx, 'pair_id'])]
                work_df.at[idx, 'llm_label'] = item.label
                work_df.at[idx, 'llm_confidence'] = item.confidence
                work_df.at[idx, 'llm_adequacy'] = item.adequacy
                work_df.at[idx, 'llm_en_language_ok'] = item.en_language_ok
                work_df.at[idx, 'llm_uz_language_ok'] = item.uz_language_ok
                work_df.at[idx, 'llm_reason'] = item.reason
                work_df.at[idx, 'llm_model'] = MODEL
                work_df.at[idx, 'llm_error'] = ''
        except Exception as exc:
            for idx in indices:
                work_df.at[idx, 'llm_error'] = str(exc)[:500]
        save_progress(work_df)
        time.sleep(REQUEST_DELAY_SECONDS)
    print('Saved:', OUTPUT_PATH)

LLM audit batches: 100%|██████████| 40/40 [08:57<00:00, 13.44s/it]

Saved: D:\dev\projects\fourlang_translation\results\data_quality\hplt_en_uz\manual_review_200_llm.csv


In [6]:
if OUTPUT_PATH.exists():
    audited = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
else:
    audited = work_df.copy()
valid = audited[audited['llm_label'].isin(['correct', 'minor', 'wrong', 'junk'])].copy()
print(f'Completed: {len(valid)}/{len(audited)}')
if len(valid):
    display(pd.crosstab(valid['sample_group'], valid['llm_label'], margins=True))
    confidence = pd.to_numeric(valid['llm_confidence'], errors='coerce')
    display(valid.assign(llm_confidence=confidence)
            .groupby('llm_label')['llm_confidence']
            .agg(['count', 'mean', 'min', 'max']).round(3))
    display(valid[['pair_id', 'sample_group', 'en', 'uz', 'llm_label', 'llm_confidence', 'llm_adequacy', 'llm_reason']].head(30))


Completed: 195/200


llm_label,correct,junk,minor,wrong,All
sample_group,,,,,
high_risk,11,4,14,68,97
random_control,65,0,13,20,98
All,76,4,27,88,195


,count,mean,min,max
llm_label,,,,
correct,76,0.937,0.8,1.0
junk,4,0.972,0.9,1.0
minor,27,0.857,0.7,1.0
wrong,88,0.957,0.8,1.0


,pair_id,sample_group,en,uz,llm_label,llm_confidence,llm_adequacy,llm_reason
0,hplt_00002652,high_risk,It's also pretty incredible that we've managed...,"Lekin, sizlardan 99%, ehtimol, biz uchun mosli...",wrong,0.9,1,乌兹别克语句子中提到了99%的用户使用官方浏览器，以及不应该出现的问题，而英语句子并没有这些...
1,hplt_00003570,high_risk,"Retrieved March 9, 2021.",Qaraldi: 8-mart 2021-yil.,minor,0.8,3,"日期翻译有误，英语中的'March 9, 2021'被译为'8-mart 2021-yil'..."
2,hplt_00002549,high_risk,Preset support.,Oldindan o'rnatilgan qo'llab -quvvatlash .,correct,0.9,4,预设支持翻译准确，意思完全一致。
3,hplt_00005357,random_control,Unlike chat rouletteson other platforms you ca...,"Ruletka chatidan farqli o'laroq, siz tasodifiy...",correct,0.9,4,句子的意思得到了准确的传达，尽管有一些小的词汇变化，但不影响整体理解。
4,hplt_00007972,random_control,The journey will take about 5 minutes.,Safar taxminan 5 daqiqa davom etadi.,correct,0.9,4,旅程大约需要5分钟的信息在两种语言中都被准确地表达了出来。
5,hplt_00002483,random_control,Another example: In case of increased body tem...,Yana bir misol: tana harorati ko'tarilgan taqd...,correct,0.9,4,句子意思忠实保留，表达自然。
6,hplt_00002750,high_risk,• double-effect HLM-1,Ichki sifatni nazorat qilish tartiblariquyidag...,wrong,0.95,1,乌兹别克语句子与英语原句意思完全不同，涉及的主题也不同。
7,hplt_00008992,random_control,The picture editor in Adobe Elements 14 is div...,Adobe Elements 14 -dagi rasm muharriri uchta r...,correct,0.85,4,句子意思忠实保留，表达自然。
8,hplt_00003037,random_control,Send your suggestions and wishes to the mail,Sizning takliflaringizni va tilaklaringizni po...,correct,0.9,4,句子意思忠实保留，表达自然。
9,hplt_00004553,high_risk,Tashkent branch,-issiqlik almashuvchisi - 1,wrong,0.95,1,乌兹别克语句子与英语原句意思完全不同，涉及的主题也不同。


In [7]:
CONFIDENCE_THRESHOLD = 0.90
if len(valid):
    confidence = pd.to_numeric(valid['llm_confidence'], errors='coerce').fillna(0.0)
    reject_mask = valid['llm_label'].isin(['wrong', 'junk']) & (confidence >= CONFIDENCE_THRESHOLD)
    keep_mask = valid['llm_label'].isin(['correct', 'minor']) & (confidence >= CONFIDENCE_THRESHOLD)
    reject = valid[reject_mask].copy()
    keep = valid[keep_mask].copy()
    needs_review = valid[~(reject_mask | keep_mask)].copy()
    reject.to_csv(OUTPUT_PATH.with_name('llm_high_confidence_reject.csv'), index=False, encoding='utf-8-sig')
    keep.to_csv(OUTPUT_PATH.with_name('llm_high_confidence_keep.csv'), index=False, encoding='utf-8-sig')
    needs_review.to_csv(OUTPUT_PATH.with_name('llm_needs_human_review.csv'), index=False, encoding='utf-8-sig')
    print('High-confidence keep:', len(keep))
    print('High-confidence reject:', len(reject))
    print('Needs human review:', len(needs_review))
    print('No original training rows were deleted.')

High-confidence keep: 83
High-confidence reject: 90
Needs human review: 22
No original training rows were deleted.
